The TIMS data used for these indicators are downloaded here at this tool:  https://tims.berkeley.edu/tools/safetypm

Choose the "SWITRS" option for the *Facility data source*.

The raw data needs to be downloaded at every level within the SACOG region and stored here:  https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FSafe%20Equitable%20Resilient%20Infrastructure%2FSafety%2FTIMS&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d


For example, in the "Fatalities" folder, there is a folder for counties, jurisdictions, MPO, and statewide.  Step through the tool and download the data one geography at a time.  For the "Counties" folder, us the *County* drop-down option and set it to "El Dorado".  Leave the *City* drop-down option set to "Show Countywide".  There should be two 6 charts total when you scroll down the webpage (2 for fatalities, 2 for serious injuries, and 2 for non-motorized), but for the "Fatalities" folder, only download a *csv* file from the first two charts.  Rename them to match the naming convention that is already in the "Counties" folder and copy/paste them to replace the old files.

Repeat this process for all counties and all jurisdictions.  Then pull the data for the SACOG MPO and CA statewide.

Then, repeat this process for the "Serious Injuries" and "Non-Motorized" folders.


Once all the data is downloaded, rerun this script (make sure that you renamed the files properly to import correctly here and run smoothly).  Check the plots to make sure that the data looks appropriate.  If the outputs look good, then uncomment all blocks of code that export the results.  Then rerun the script and check the outputs on SharePoint.

In [ ]:


# TIMS does not have an API available.  Data were manually downloaded from here https://tims.berkeley.edu/ and stored on SharePoint


EXPORT=False


import pandas as pd
import os
from pathlib import Path
import plotly.express as px
from IPython.display import display
pd.set_option('display.max_columns', None)


PATH_GIT = Path.cwd().parent.parent
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_GIT / 'Data' / 'TIMS' / 'config'

# SharePoint OneDrive paths
PATH_MAIN = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data'
PATH_OUT  = PATH_MAIN / 'Safe Equitable Resilient Infrastructure' / 'Safety'
PATH_TIMS = PATH_OUT / 'TIMS'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")

FILE_ABOUT = PATH_MAIN.parent / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'


import sys
sys.path.append(str(PATH_CONFIG0))
import functions as func
import plot as pt

sys.path.append(str(PATH_CONFIG))
import tims



In [ ]:

year_start = 2012
year_end = 2024
years_to_import = range(year_start, year_end+1)

categories = [cat for cat in os.listdir(PATH_TIMS) if '.xlsx' not in cat]
categories.remove('Bicyclists vs Pedestrians')

print(categories, years_to_import)


In [ ]:


# Import data at the jurisdictions level
df_tims1 = tims.jurisdiction(categories, years_to_import)
display(df_tims1.head())


# Subset all TIMS data into different categories
list_id = ['County', 'Jurisdiction', 'Year']

df_tims1_fat = df_tims1[list_id + [              'Fatalities',               'Fatalities_5 Year Rolling Average']]
df_tims1_ser = df_tims1[list_id + [        'Serious injuries',         'Serious injuries_5 Year Rolling Average']]
df_tims1_non = df_tims1[list_id + ['Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                 , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]

df_tims1_fat100 = df_tims1[list_id + [      'Fatalities 100 mvmt',      'Fatalities 100 mvmt_5 Year Rolling Average']]
df_tims1_ser100 = df_tims1[list_id + ['Serious injuries 100 mvm' , 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Set columns of jurisdictions by counties
df_tims1_cols = df_tims1_fat.pivot_table(index = 'Year'
                                        , columns = ['County', 'Jurisdiction']
                                        , values = ['Fatalities', 'Fatalities_5 Year Rolling Average']).reset_index()
cols = [col[2] for col in df_tims1_cols.columns][1:]
cols = ['Year'] + cols

# Fatalities and Fatalities 100/MVMT
df_tims1_fat_2      = df_tims1_fat   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities'                                ).reset_index()
df_tims1_fat_2_5    = df_tims1_fat   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities_5 Year Rolling Average'         ).reset_index()
df_tims1_fat100_2   = df_tims1_fat100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities 100 mvmt'                       ).reset_index()
df_tims1_fat100_2_5 = df_tims1_fat100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Fatalities 100 mvmt_5 Year Rolling Average').reset_index()

# Serious Injuries and Serious Injuries 100/MVMT
df_tims1_ser_2      = df_tims1_ser   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries'                               ).reset_index()
df_tims1_ser_2_5    = df_tims1_ser   .pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries_5 Year Rolling Average'        ).reset_index()
df_tims1_ser100_2   = df_tims1_ser100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries 100 mvm'                       ).reset_index()
df_tims1_ser100_2_5 = df_tims1_ser100.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Serious injuries 100 mvm_5 Year Rolling Average').reset_index()

# Non-Motorized Fatalities and Serious Injuries
df_tims1_non_fat_2   = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized fatalities'                       ).reset_index()
df_tims1_non_fat_2_5 = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized fatalities_5 Year Rolling Average').reset_index()
df_tims1_non_si_2    = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized serious in'                       ).reset_index()
df_tims1_non_si_2_5  = df_tims1_non.pivot_table(index = 'Year', columns = 'Jurisdiction', values = 'Non motorized serious in_5 Year Rolling Average').reset_index()

# Reorganize columns
df_tims1_fat_2       = df_tims1_fat_2      [cols]
df_tims1_fat_2_5     = df_tims1_fat_2_5    [cols]
df_tims1_fat100_2    = df_tims1_fat100_2   [cols]
df_tims1_fat100_2_5  = df_tims1_fat100_2_5 [cols]
df_tims1_ser_2       = df_tims1_ser_2      [cols]
df_tims1_ser_2_5     = df_tims1_ser_2_5    [cols]
df_tims1_ser100_2    = df_tims1_ser100_2   [cols]
df_tims1_ser100_2_5  = df_tims1_ser100_2_5 [cols]
df_tims1_non_fat_2   = df_tims1_non_fat_2  [cols]
df_tims1_non_fat_2_5 = df_tims1_non_fat_2_5[cols]
df_tims1_non_si_2    = df_tims1_non_si_2   [cols]
df_tims1_non_si_2_5  = df_tims1_non_si_2_5 [cols]

display(df_tims1.head())


if EXPORT:
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data by Jurisdiction.xlsx', engine='xlsxwriter') as writer:
        df_tims1            .to_excel(writer, index = False, sheet_name = 'All'                            )
        df_tims1_fat_2      .to_excel(writer, index = False, sheet_name = 'Fatalities'                     )
        df_tims1_fat_2_5    .to_excel(writer, index = False, sheet_name = 'Fatalities 5 year'              )
        df_tims1_fat100_2   .to_excel(writer, index = False, sheet_name = 'Fatalities rate'                )
        df_tims1_fat100_2_5 .to_excel(writer, index = False, sheet_name = 'Fatalities rate 5 year'         )
        df_tims1_ser_2      .to_excel(writer, index = False, sheet_name = 'Serious injuries'               )
        df_tims1_ser_2_5    .to_excel(writer, index = False, sheet_name = 'Serious injuries 5 year'        )
        df_tims1_ser100_2   .to_excel(writer, index = False, sheet_name = 'Serious injuries rate'          )
        df_tims1_ser100_2_5 .to_excel(writer, index = False, sheet_name = 'Serious injuries rate 5 year'   )
        df_tims1_non_fat_2  .to_excel(writer, index = False, sheet_name = 'Non motorized fatalities'       )
        df_tims1_non_fat_2_5.to_excel(writer, index = False, sheet_name = 'Non motorized fatalities 5 year')
        df_tims1_non_si_2   .to_excel(writer, index = False, sheet_name = 'Non motorized serious in'       )
        df_tims1_non_si_2_5 .to_excel(writer, index = False, sheet_name = 'Non motorized serious in 5 year')




In [ ]:




# Import data at the county level
df_tims2 = tims.county(categories, years_to_import)
display(df_tims2.head())



# Subset all TIMS data into different categories
list_id = ['County', 'Year']

df_tims2_fat = df_tims2[list_id + [              'Fatalities',               'Fatalities_5 Year Rolling Average']]
df_tims2_ser = df_tims2[list_id + [        'Serious injuries',         'Serious injuries_5 Year Rolling Average']]
df_tims2_non = df_tims2[list_id + ['Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                 , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]

df_tims2_fat100 = df_tims2[list_id + [      'Fatalities 100 mvmt',      'Fatalities 100 mvmt_5 Year Rolling Average']]
df_tims2_ser100 = df_tims2[list_id + ['Serious injuries 100 mvm' , 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Set columns of jurisdictions by counties
df_tims2_cols = df_tims2_fat.pivot_table(index = 'Year'
                                        , columns = 'County'
                                        , values = ['Fatalities', 'Fatalities_5 Year Rolling Average']).reset_index()
cols = [col[1] for col in df_tims2_cols.columns][1:]
cols = ['Year'] + cols

# Fatalities and Fatalities 100/MVMT
df_tims2_fat_2      = df_tims2_fat   .pivot_table(index = 'Year', columns = 'County', values = 'Fatalities'                                ).reset_index()
df_tims2_fat_2_5    = df_tims2_fat   .pivot_table(index = 'Year', columns = 'County', values = 'Fatalities_5 Year Rolling Average'         ).reset_index()
df_tims2_fat100_2   = df_tims2_fat100.pivot_table(index = 'Year', columns = 'County', values = 'Fatalities 100 mvmt'                       ).reset_index()
df_tims2_fat100_2_5 = df_tims2_fat100.pivot_table(index = 'Year', columns = 'County', values = 'Fatalities 100 mvmt_5 Year Rolling Average').reset_index()

# Serious Injuries and Serious Injuries 100/MVMT
df_tims2_ser_2      = df_tims2_ser   .pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries'                               ).reset_index()
df_tims2_ser_2_5    = df_tims2_ser   .pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries_5 Year Rolling Average'        ).reset_index()
df_tims2_ser100_2   = df_tims2_ser100.pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries 100 mvm'                       ).reset_index()
df_tims2_ser100_2_5 = df_tims2_ser100.pivot_table(index = 'Year', columns = 'County', values = 'Serious injuries 100 mvm_5 Year Rolling Average').reset_index()

# Non-Motorized Fatalities and Serious Injuries
df_tims2_non_fat_2   = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized fatalities'                       ).reset_index()
df_tims2_non_fat_2_5 = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized fatalities_5 Year Rolling Average').reset_index()
df_tims2_non_si_2    = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized serious in'                       ).reset_index()
df_tims2_non_si_2_5  = df_tims2_non.pivot_table(index = 'Year', columns = 'County', values = 'Non motorized serious in_5 Year Rolling Average').reset_index()

# Reorganize columns
df_tims2_fat_2       = df_tims2_fat_2      [cols]
df_tims2_fat_2_5     = df_tims2_fat_2_5    [cols]
df_tims2_fat100_2    = df_tims2_fat100_2   [cols]
df_tims2_fat100_2_5  = df_tims2_fat100_2_5 [cols]
df_tims2_ser_2       = df_tims2_ser_2      [cols]
df_tims2_ser_2_5     = df_tims2_ser_2_5    [cols]
df_tims2_ser100_2    = df_tims2_ser100_2   [cols]
df_tims2_ser100_2_5  = df_tims2_ser100_2_5 [cols]
df_tims2_non_fat_2   = df_tims2_non_fat_2  [cols]
df_tims2_non_fat_2_5 = df_tims2_non_fat_2_5[cols]
df_tims2_non_si_2    = df_tims2_non_si_2   [cols]
df_tims2_non_si_2_5  = df_tims2_non_si_2_5 [cols]

display(df_tims2.head())



if EXPORT:
    # Counties
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data by County.xlsx', engine='xlsxwriter') as writer:
        df_tims2            .to_excel(writer, index = False, sheet_name = 'All'                            )
        df_tims2_fat_2      .to_excel(writer, index = False, sheet_name = 'Fatalities'                     )
        df_tims2_fat_2_5    .to_excel(writer, index = False, sheet_name = 'Fatalities 5 year'              )
        df_tims2_fat100_2   .to_excel(writer, index = False, sheet_name = 'Fatalities rate'                )
        df_tims2_fat100_2_5 .to_excel(writer, index = False, sheet_name = 'Fatalities rate 5 year'         )
        df_tims2_ser_2      .to_excel(writer, index = False, sheet_name = 'Serious injuries'               )
        df_tims2_ser_2_5    .to_excel(writer, index = False, sheet_name = 'Serious injuries 5 year'        )
        df_tims2_ser100_2   .to_excel(writer, index = False, sheet_name = 'Serious injuries rate'          )
        df_tims2_ser100_2_5 .to_excel(writer, index = False, sheet_name = 'Serious injuries rate 5 year'   )
        df_tims2_non_fat_2  .to_excel(writer, index = False, sheet_name = 'Non motorized fatalities'       )
        df_tims2_non_fat_2_5.to_excel(writer, index = False, sheet_name = 'Non motorized fatalities 5 year')
        df_tims2_non_si_2   .to_excel(writer, index = False, sheet_name = 'Non motorized serious in'       )
        df_tims2_non_si_2_5 .to_excel(writer, index = False, sheet_name = 'Non motorized serious in 5 year')



In [ ]:
path_plots = PATH_OUT / 'Safety_1 Collision Rates' / 'plots'
df_plot = df_tims2.copy()

df_plot = df_plot[['County', 'Year', 'Fatalities 100 mvmt', 'Serious injuries 100 mvm']]
df_plot = df_plot.rename(columns = {'Fatalities 100 mvmt':'Fatalities', 'Serious injuries 100 mvm':'Serious Injuries'})
df_plot = pd.melt(df_plot, id_vars = ['County', 'Year'], var_name = 'Category', value_name = 'value')
df_plot = df_plot.drop_duplicates()

display(df_plot.head())

fig = px.line(df_plot, x = 'Year', y = 'value', color = 'County', line_dash = 'Category', markers = False)
fig.update_layout(legend_title=None, title='TIMS Fatalities vs Serious Injuries by County (Per 1000 MVMT)')
fig.show()

df_plot = df_plot[df_plot['Year'].isin([2015, 2019, 2023])]
fig = px.bar(df_plot, x = 'Category', y = 'value', color = 'County', barmode = 'group', facet_col = 'Year')
fig.update_layout(legend_title=None, title='TIMS Fatalities vs Serious Injuries by County (Per 1000 MVMT)')
fig.show()

In [ ]:
path_plots = PATH_OUT / 'Safety_2 Non-Motorized' / 'plots'
df_plot = df_tims2.copy()

df_plot = df_plot[['County', 'Year', 'Non motorized fatalities', 'Non motorized serious in']]
df_plot = df_plot.rename(columns = {'Non motorized fatalities':'Fatalities', 'Non motorized serious in':'Serious Injuries'})
df_plot = pd.melt(df_plot, id_vars = ['County', 'Year'], var_name = 'Category', value_name = 'value')
df_plot = df_plot.drop_duplicates()

display(df_plot.head())

fig = px.line(df_plot, x = 'Year', y = 'value', color = 'County', line_dash = 'Category', markers = False)
fig.update_layout(legend_title=None, title='TIMS Non-Motorized Fatalities vs Serious Injuries by County')
fig.show()

df_plot = df_plot[df_plot['Year'].isin([2015, 2019, 2023])]
fig = px.bar(df_plot, x = 'Category', y = 'value', color = 'County', barmode = 'group', facet_col = 'Year')
fig.update_layout(legend_title=None, title='TIMS Non-Motorized Fatalities vs Serious Injuries by County')
fig.show()

In [ ]:


# Import data at the mpo level
df_tims3 = tims.mpo(categories, years_to_import)
display(df_tims3.head())

if EXPORT:
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data by MPO.xlsx', engine='xlsxwriter') as writer:
        df_tims3.to_excel(writer, index=False, sheet_name='All')


# Import data statewide
df_tims4 = tims.statewide(categories, years_to_import)
display(df_tims4.head())

if EXPORT:
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data Statewide.xlsx', engine='xlsxwriter') as writer:
        df_tims4.to_excel(writer, index=False, sheet_name='Statewide')



In [ ]:

# Safety_1 Collision Rates
df_safety1_1 = df_tims1[['County', 'Jurisdiction', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                                         , 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                                         , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_2 = df_tims2[['County', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                         , 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                         , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                         , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_3 = df_tims3[['MPO', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                      , 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                      , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                      , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]
df_safety1_4 = df_tims4[['State', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                        , 'Fatalities 100 mvmt'     , 'Fatalities 100 mvmt_5 Year Rolling Average'
                                        , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                        , 'Serious injuries 100 mvm', 'Serious injuries 100 mvm_5 Year Rolling Average']]

# Safety_2 Non-Motorized
df_safety2_1 = df_tims1[['County', 'Jurisdiction', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                         , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_2 = df_tims2[['County', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                         , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_3 = df_tims3[['MPO', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                      , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_4 = df_tims4[['State', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                        , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]



if EXPORT:

    # Update overall about documentations workbook
    sample_type = 'TIMS'
    indicators = ['Safety_1', 'Safety_2']

    for indicator in indicators:
        df_about = func.write_about(sample_type   = sample_type
                                    , indicator   = indicator
                                    , year_start  = year_start
                                    , year_end    = year_end)
        with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
            df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    sample_type = 'TIMS'
    categories = ['Collision Rates', 'Non-Motorized']

    paths = [PATH_OUT, PATH_SERVER]

    for path in paths:
        for category in categories:
            if category == 'Collision Rates':
                indicator = 'Safety_1'
                dt_geo = {
                    'Jurisdictions': df_safety1_1,
                    'Counties': df_safety1_2,
                    'MPO': df_safety1_3,
                    'Statewide': df_safety1_4
                    }
            if category == 'Non-Motorized':
                indicator = 'Safety_2'
                dt_geo = {
                    'Jurisdictions': df_safety2_1,
                    'Counties': df_safety2_2,
                    'MPO': df_safety2_3,
                    'Statewide': df_safety2_4
                    }
            for geography, df in dt_geo.items():
                df_about = func.write_about(sample_type  = sample_type
                                            , indicator  = indicator
                                            , year_start = year_start
                                            , year_end   = year_end
                                            , geography  = geography)
                if path == PATH_OUT:
                    file_out = path / f'{indicator} {category}' / f'{indicator} {geography} {sample_type}.xlsx'
                if path == PATH_SERVER:
                    file_out = path / f'{indicator} {geography} {sample_type}.xlsx'
                print('Excel files exported here: ' + str(file_out))
                with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
                    df_about.to_excel(writer, index=False, sheet_name='About'  , header=False)
                    df      .to_excel(writer, index=False, sheet_name=geography              )

                
